# Probability inequalities

## From Markov to Hoeffding — bounding what you cannot compute

In probability and statistics, we often want to know how likely a random variable is to be far from its expectation. Probability inequalities provide rigorous upper bounds on tail probabilities given limited distributional information.

This notebook generates the figures for the blog post. The PNGs are saved into `content/blog/prob-inequalities/` next to `index.md` (Hugo page bundle). After regenerating, run **`hugo`** from the repository root.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless backend

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

for _style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"):
    try:
        plt.style.use(_style)
        break
    except OSError:
        continue
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RNG = np.random.default_rng(20260409)


def locate_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "content").exists() and (cwd / "notebooks").exists():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "content").exists():
        return cwd.parent
    for parent in [cwd, *cwd.parents]:
        if (parent / "content").exists() and (parent / "notebooks").exists():
            return parent
        if (parent / "hugo.toml").exists() and (parent / "content").exists():
            return parent
    raise RuntimeError("Could not locate repository root.")


ROOT = locate_root()
OUTDIR = ROOT / "content" / "blog" / "prob-inequalities"
OUTDIR.mkdir(parents=True, exist_ok=True)


def savefig(name):
    path = (OUTDIR / name).resolve()
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", dpi=180, facecolor="white")
    plt.close()
    print(path.relative_to(ROOT))
    return path


ROOT, OUTDIR

## Figure 1 — Markov's inequality illustration

We plot the true tail probability $\mathbb{P}(Z \geq t)$ for a $\chi^2_3$ random variable alongside the Markov bound $\mathbb{E}[Z]/t = 3/t$. Markov's bound is always valid but becomes increasingly loose in the tail.

In [ ]:
df_chi2 = 3
t_vals = np.linspace(0.1, 15, 500)
true_tail = stats.chi2.sf(t_vals, df=df_chi2)
markov_bound = df_chi2 / t_vals

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_vals, true_tail, color="#2c3e50", linewidth=2, label=r"True $\mathbb{P}(Z \geq t)$")
ax.plot(
    t_vals,
    np.minimum(markov_bound, 1.05),
    color="#e74c3c",
    linewidth=2,
    linestyle="--",
    label=r"Markov bound $\mathbb{E}[Z]/t$",
)
ax.fill_between(
    t_vals,
    true_tail,
    np.minimum(markov_bound, 1.05),
    alpha=0.12,
    color="#e74c3c",
    label="Gap (bound looseness)",
)
ax.set_xlabel(r"$t$", fontsize=13)
ax.set_ylabel("Probability / Bound", fontsize=12)
ax.set_ylim(-0.02, 1.08)
ax.set_xlim(0, 15)
ax.legend(fontsize=11, loc="upper right")
ax.set_title(r"Markov's inequality for $Z \sim \chi^2_3$", fontsize=13)

savefig("markov-illustration.png")

## Figure 2 — Comparing tightness of bounds

For $X \sim \text{Binom}(n, 1/2)$, we compare the true tail probability $\mathbb{P}(X \geq \lceil 3n/4 \rceil)$ with four upper bounds:

- **Markov:** $2/3$ (constant)
- **Chebyshev:** $4/n$
- **Chernoff:** $(16/27)^{n/4}$
- **Hoeffding:** $e^{-n/8}$

In [ ]:
n_vals = np.arange(1, 81)

true_prob = np.array(
    [stats.binom.sf(int(np.ceil(3 * n / 4)) - 1, n, 0.5) for n in n_vals]
)
markov = np.full_like(n_vals, 2.0 / 3, dtype=float)
chebyshev = 4.0 / n_vals
chernoff = (16.0 / 27) ** (n_vals / 4.0)
hoeffding = np.exp(-n_vals / 8.0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(n_vals, true_prob, "o-", color="#2c3e50", markersize=3, linewidth=1.5, label="True probability")
ax.semilogy(n_vals, markov, "--", color="#e74c3c", linewidth=2, label="Markov: $2/3$")
ax.semilogy(n_vals, chebyshev, "--", color="#e67e22", linewidth=2, label="Chebyshev: $4/n$")
ax.semilogy(n_vals, chernoff, "--", color="#27ae60", linewidth=2, label=r"Chernoff: $(16/27)^{n/4}$")
ax.semilogy(n_vals, hoeffding, "--", color="#8e44ad", linewidth=2, label=r"Hoeffding: $e^{-n/8}$")

ax.set_xlabel(r"$n$", fontsize=13)
ax.set_ylabel(r"$\mathbb{P}(X \geq \lceil 3n/4 \rceil)$", fontsize=13)
ax.set_title(r"Tail bounds for $X \sim \mathrm{Binom}(n,\, 1/2)$", fontsize=13)
ax.legend(fontsize=10, loc="lower left")
ax.set_ylim(1e-12, 2)
ax.set_xlim(1, 80)

savefig("tightness-comparison.png")